# Fase 1: Gaming vs Academic Performance

**Integrantes:** ________________________________

**Objetivo del notebook:** cargar el dataset original, documentar y ejecutar una limpieza y transformación reproducible para estudiar, en fases posteriores, la asociación entre horas diarias de videojuegos y rendimiento académico, considerando horas de estudio y sueño. El dataset es simulado según su autor.

## Índice CRISP-DM

0. Configuración y carga — **COMPLETA**
1. Comprensión del problema — **PENDIENTE**
2. Comprensión y selección de los datos — **PENDIENTE**
3. Limpieza y transformación — **COMPLETA**
4. Análisis univariado y bivariado — **PENDIENTE**
5. Análisis descriptivo y exploratorio — **PENDIENTE**
6. Conclusión de la Fase 1 — **PENDIENTE**

## Sección 0. Configuración y carga
Se definen rutas relativas al directorio de trabajo `notebooks`, se cargan las librerías permitidas y se conserva `df_raw` sin modificar para el perfilado posterior.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

RAW_PATH = Path('../data/raw/Gaming_Academic_Performance_updated.csv')
CLEAN_PATH = Path('../data/processed/gaming_academic_clean.csv')
BITACORA_PATH = Path('../output/bitacora_limpieza.csv')
TOPE_NOTAS = 100

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
BITACORA_PATH.parent.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW_PATH)
n0 = len(df_raw)
hash_sha256 = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()

print(f'Dimensiones originales: {df_raw.shape}')
print('Tipos originales:')
print(df_raw.dtypes)
print(f'Hash SHA-256: {hash_sha256}')
print(f'n0: {n0}')

# Toda la limpieza se hace sobre esta copia; df_raw permanece sin modificar.
df = df_raw.copy()
registros_bitacora = []

def registrar(n_paso, paso, columna, revision_o_problema, registros_afectados, decision, fundamento):
    registros_bitacora.append({
        'n_paso': n_paso,
        'paso': paso,
        'columna': columna,
        'revision_o_problema': revision_o_problema,
        'registros_afectados': registros_afectados,
        'decision': decision,
        'fundamento': fundamento,
        'filas_despues': len(df)
    })

pandas: 2.3.3
numpy: 2.5.3
Dimensiones originales: (8000, 14)
Tipos originales:
student_id            int64
age                   int64
gender               object
gaming_hours        float64
study_hours         float64
sleep_hours         float64
attendance          float64
gaming_genre         object
social_activity     float64
device_usage        float64
reaction_time_ms    float64
addiction_score     float64
stress_level         object
grades              float64
dtype: object
Hash SHA-256: d3826b37ee541811f34ae5fbaae6a34bd59af224ab09999b9f92d1c63281ba26
n0: 8000


## Sección 1. Comprensión del problema
PENDIENTE: se redacta en el documento de comprensión del problema y se resume aquí.

## Sección 2. Comprensión y selección de los datos
PENDIENTE: diccionario de datos, perfilado del dataset original (`df_raw`), evaluación de calidad, selección justificada y documentación de la fuente.

## Sección 3. Limpieza y transformación
Las revisiones se registran incluso cuando no encuentran problemas. No se eliminan filas ni se imputan valores.

### 3.1 Revisiones previas
Se revisan faltantes, duplicados, unicidad, columnas repetidas, categorías, rangos, atípicos y consistencias sin modificar los datos.

In [2]:
# Faltantes por columna.
faltantes = df.isna().sum()
for columna, cantidad in faltantes.items():
    registrar('3.1', 'Revisiones previas', columna, 'Valores faltantes', int(cantidad), 'No imputar', 'Se conservan las filas y se documentan los faltantes.')

duplicadas_completas = int(df.duplicated().sum())
registrar('3.1', 'Revisiones previas', 'todas', 'Filas duplicadas completas', duplicadas_completas, 'No eliminar filas', 'La consigna exige conservar las 8000 filas.')
duplicadas_sin_id = int(df.drop(columns=['student_id']).duplicated().sum())
registrar('3.1', 'Revisiones previas', 'todas excepto student_id', 'Filas duplicadas ignorando student_id', duplicadas_sin_id, 'No eliminar filas', 'La revisión informa calidad sin alterar el dataset.')

id_notnull = int(df['student_id'].notna().sum())
id_nunique = int(df['student_id'].nunique())
id_unico = bool(df['student_id'].is_unique)
fallos_id = (n0 - id_notnull) + (n0 - id_nunique) + int(not id_unico)
registrar('3.1', 'Revisiones previas', 'student_id', 'Unicidad: notnull, nunique y is_unique', fallos_id, 'Conservar para revisar y descartar después', f'notnull={id_notnull}; nunique={id_nunique}; is_unique={id_unico}.')

nombres_duplicados = int(pd.Index(df.columns).duplicated().sum())
columnas_duplicadas = int(df.T.duplicated().sum())
registrar('3.1', 'Revisiones previas', 'nombres de columnas', 'Nombres de columna duplicados', nombres_duplicados, 'No modificar', 'Los nombres deben ser únicos.')
registrar('3.1', 'Revisiones previas', 'columnas', 'Columnas con contenido idéntico', columnas_duplicadas, 'No modificar', 'Se evita conservar información repetida sin cambiar el original.')

espacios_por_categoria = {}
variantes_por_categoria = {}
for columna in ['gender', 'gaming_genre', 'stress_level']:
    valores = df[columna].dropna().astype(str)
    con_espacios = int((valores != valores.str.strip()).sum())
    grupos_mayusculas = valores.groupby(valores.str.lower()).nunique()
    variantes = int((grupos_mayusculas > 1).sum())
    espacios_por_categoria[columna] = con_espacios
    variantes_por_categoria[columna] = variantes
    registrar('3.1', 'Revisiones previas', columna, 'Categorías, espacios sobrantes y variantes de mayúsculas', con_espacios + variantes, f'Valores únicos: {sorted(valores.unique().tolist())}', 'Se documentan las categorías antes de normalizar tipos.')
    print(f'{columna}: valores={sorted(valores.unique().tolist())}; espacios={con_espacios}; grupos_con_variantes_mayusculas={variantes}')

columnas_numericas = df.select_dtypes(include=np.number).columns
atipicos_por_columna = {}
for columna in columnas_numericas:
    minimo = df[columna].min()
    maximo = df[columna].max()
    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)
    iqr = q3 - q1
    atipicos = int(((df[columna] < q1 - 1.5 * iqr) | (df[columna] > q3 + 1.5 * iqr)).sum())
    atipicos_por_columna[columna] = atipicos
    registrar('3.1', 'Revisiones previas', columna, 'Rango y atípicos por regla 1.5*IQR', atipicos, f'mín={minimo}; máx={maximo}', 'Se informa el rango y no se eliminan atípicos.')
    print(f'{columna}: mínimo={minimo}; máximo={maximo}; atípicos={atipicos}')

consistencias = {
    'device_usage >= gaming_hours': int((df['device_usage'] < df['gaming_hours']).sum()),
    'gaming_hours + study_hours + sleep_hours <= 24': int((df[['gaming_hours', 'study_hours', 'sleep_hours']].sum(axis=1) > 24).sum()),
    'attendance fuera de [0, 100]': int((~df['attendance'].between(0, 100)).sum()),
    'grades negativas': int((df['grades'] < 0).sum())
}
for revision, cantidad in consistencias.items():
    registrar('3.1', 'Revisiones previas', 'consistencia', revision, cantidad, 'Revisar sin eliminar filas', 'Las reglas de consistencia se informan antes de transformar.')
print('Consistencias:', consistencias)

def comparar(nombre, observado, esperado, tolerancia=0):
    estado = 'OK' if abs(observado - esperado) <= tolerancia else 'REVISAR'
    print(f'{nombre}: observado={observado}; esperado={esperado}; {estado}')

comparar('faltantes totales', int(faltantes.sum()), 0)
comparar('duplicadas completas', duplicadas_completas, 0)
comparar('duplicadas ignorando student_id', duplicadas_sin_id, 0)
comparar('student_id distintos', id_nunique, n0)
comparar('student_id único', int(id_unico), 1)
comparar('nombres de columna duplicados', nombres_duplicados, 0)
comparar('columnas con contenido idéntico', columnas_duplicadas, 0)
for columna in espacios_por_categoria:
    comparar(f'{columna} con espacios sobrantes', espacios_por_categoria[columna], 0)
    comparar(f'{columna} con variantes de mayúsculas', variantes_por_categoria[columna], 0)
comparar('atípicos totales', sum(atipicos_por_columna.values()), 0)
comparar('device_usage menor que gaming_hours', consistencias['device_usage >= gaming_hours'], 0)

gender: valores=['Female', 'Male', 'Other']; espacios=0; grupos_con_variantes_mayusculas=0
gaming_genre: valores=['Casual', 'FPS', 'RPG']; espacios=0; grupos_con_variantes_mayusculas=0
stress_level: valores=['High', 'Low', 'Medium']; espacios=0; grupos_con_variantes_mayusculas=0
student_id: mínimo=1; máximo=8000; atípicos=0
age: mínimo=16; máximo=24; atípicos=0
gaming_hours: mínimo=0.0; máximo=8.0; atípicos=0
study_hours: mínimo=1.0; máximo=10.0; atípicos=0
sleep_hours: mínimo=4.0; máximo=9.0; atípicos=0
attendance: mínimo=60.0; máximo=100.0; atípicos=0
social_activity: mínimo=0.0; máximo=5.0; atípicos=0
device_usage: mínimo=1.1; máximo=13.95; atípicos=0
reaction_time_ms: mínimo=183.26; máximo=347.87; atípicos=0
addiction_score: mínimo=-4.51; máximo=23.16; atípicos=0
grades: mínimo=0.0; máximo=118.63293578685628; atípicos=0
Consistencias: {'device_usage >= gaming_hours': 0, 'gaming_hours + study_hours + sleep_hours <= 24': 0, 'attendance fuera de [0, 100]': 0, 'grades negativas': 0}
fa

### 3.2 Indicador de registros ajustados
Se identifica antes del redondeo si `grades` tiene más de dos decimales. La nota de Kaggle indica que la versión `updated` ajustó registros con horas totales implausibles; la relación entre esos decimales adicionales y los ajustes es una **hipótesis**, no un hecho confirmado.

In [3]:
flag_ajustado = (df['grades'] - df['grades'].round(2)).abs() > 1e-9
df['flag_ajustado'] = flag_ajustado.astype(bool)
cantidad_ajustados = int(flag_ajustado.sum())
suma_horas = df[['gaming_hours', 'study_hours', 'sleep_hours']].sum(axis=1)
promedio_ajustadas = suma_horas[flag_ajustado].mean()
promedio_no_ajustadas = suma_horas[~flag_ajustado].mean()
print(f'Registros con más de dos decimales en grades: {cantidad_ajustados}')
print(f'Media de horas totales en filas ajustadas: {promedio_ajustadas:.4f}')
print(f'Media de horas totales en filas no ajustadas: {promedio_no_ajustadas:.4f}')
registrar('3.2', 'Indicador de registros ajustados', 'grades', 'Más de dos decimales antes del redondeo', cantidad_ajustados, 'Crear flag_ajustado', 'La relación con ajustes de horas es una hipótesis y se conserva como indicador.')
comparar('grades con más de dos decimales', cantidad_ajustados, 1671, tolerancia=100)

Registros con más de dos decimales en grades: 1671
Media de horas totales en filas ajustadas: 18.8327
Media de horas totales en filas no ajustadas: 15.2625
grades con más de dos decimales: observado=1671; esperado=1671; OK


### 3.3 Limpieza de `grades`
Se conserva la nota original, se marca cada valor superior a 100 y se aplica un tope de 100. Se asume que `grades` está en escala 0-100; el posible efecto techo queda como limitación para el análisis posterior.

In [4]:
grades_original = df['grades'].copy()
flag_grades_gt100 = df['grades'] > TOPE_NOTAS
cantidad_gt100 = int(flag_grades_gt100.sum())
cantidad_100 = int((df['grades'] == 100).sum())
cantidad_0 = int((df['grades'] == 0).sum())
print(f'Notas mayores que 100: {cantidad_gt100}')
print(f'Notas exactamente iguales a 100: {cantidad_100}')
print(f'Notas exactamente iguales a 0: {cantidad_0}')
print(f'Máximo original de grades: {grades_original.max()}')
comparar('máximo original de grades', float(grades_original.max()), 118.63, tolerancia=0.01)

df['grades_original'] = grades_original
df['flag_grades_gt100'] = flag_grades_gt100.astype(bool)
df['grades'] = df['grades'].clip(upper=TOPE_NOTAS)
registrar('3.3', 'Limpieza de grades', 'grades', 'Valores superiores al tope de notas', cantidad_gt100, 'Aplicar clip superior en 100 y conservar grades_original', 'Conserva las 8000 filas, sigue la escala 0-100 asumida y permite análisis de sensibilidad posterior.')
comparar('grades mayores que 100', cantidad_gt100, 134, tolerancia=20)
comparar('grades iguales a 100', cantidad_100, 439, tolerancia=20)
comparar('grades iguales a 0', cantidad_0, 1)

Notas mayores que 100: 134
Notas exactamente iguales a 100: 439
Notas exactamente iguales a 0: 1
Máximo original de grades: 118.63293578685628
máximo original de grades: observado=118.63293578685628; esperado=118.63; OK
grades mayores que 100: observado=134; esperado=134; OK
grades iguales a 100: observado=439; esperado=439; OK
grades iguales a 0: observado=1; esperado=1; OK


### 3.4 Tipos y categorías
Se eliminan espacios externos en las categorías, se convierten `gender` y `stress_level` a categorías y se define el orden lógico de `stress_level`. `gaming_genre` se normaliza mientras todavía existe.

In [5]:
for columna in ['gender', 'stress_level', 'gaming_genre']:
    antes = df[columna].copy()
    df[columna] = df[columna].str.strip()
    cambiaron = int((antes != df[columna]).sum())
    registrar('3.4', 'Tipos y categorías', columna, 'Espacios externos', cambiaron, 'Aplicar str.strip()', 'Normaliza categorías sin cambiar su significado.')
    print(f'{columna}: valores modificados por strip={cambiaron}')

df['gender'] = df['gender'].astype('category')
stress_categorias = ['Low', 'Medium', 'High']
df['stress_level'] = pd.Categorical(df['stress_level'], categories=stress_categorias, ordered=True)
age_entero = bool(np.all(df['age'].dropna() == df['age'].dropna().astype(int)))
registrar('3.4', 'Tipos y categorías', 'age', 'Verificación de tipo entero', 0 if age_entero else 1, 'Conservar tipo entero si se cumple', 'La edad está definida como años enteros.')
print(f'age es entero: {age_entero}')

gender: valores modificados por strip=0
stress_level: valores modificados por strip=0
gaming_genre: valores modificados por strip=0
age es entero: True


### 3.5 Variables derivadas
Se asume que `device_usage` incluye el tiempo de juego. Por eso se crea `uso_no_gaming` para contrastar juego y tiempo de pantalla general. `device_usage`, `gaming_hours` y `uso_no_gaming` no deben entrar juntas en una regresión porque tienen dependencia lineal perfecta.

`gaming_cat` usa terciles de la muestra. Como alternativa podrían usarse cortes fijos en 2 y 5 horas; los terciles son una decisión que debe justificarse en el análisis.

In [6]:
device_menor = int((df['device_usage'] < df['gaming_hours']).sum())
print(f'Filas con device_usage < gaming_hours: {device_menor}')
registrar('3.5', 'Variables derivadas', 'uso_no_gaming', 'device_usage >= gaming_hours', device_menor, 'Crear diferencia solo si la consistencia se mantiene', 'Supuesto: device_usage incluye gaming_hours.')

df['uso_no_gaming'] = df['device_usage'] - df['gaming_hours']
df['gaming_cat'], cortes_gaming = pd.qcut(df['gaming_hours'], q=3, labels=['Bajo', 'Medio', 'Alto'], retbins=True)
df['gaming_cat'] = pd.Categorical(df['gaming_cat'], categories=['Bajo', 'Medio', 'Alto'], ordered=True)
tamanos_gaming = df['gaming_cat'].value_counts(sort=False)
print(f'Cortes de gaming_hours: {cortes_gaming}')
print('Tamaño de cada grupo:')
print(tamanos_gaming)
comparar('cortes de gaming_hours: tercil 1', float(cortes_gaming[1]), 2.8, tolerancia=0.1)
comparar('cortes de gaming_hours: tercil 2', float(cortes_gaming[2]), 5.46, tolerancia=0.1)
registrar('3.5', 'Variables derivadas', 'gaming_cat', 'Cortes por terciles', len(cortes_gaming) - 1, f'Cortes: {cortes_gaming.tolist()}', 'Agrupa en Bajo, Medio y Alto para comparación descriptiva; los cortes fijos 2 y 5 horas son una alternativa.')

Filas con device_usage < gaming_hours: 0
Cortes de gaming_hours: [0.   2.8  5.46 8.  ]
Tamaño de cada grupo:
gaming_cat
Bajo     2671
Medio    2667
Alto     2662
Name: count, dtype: int64
cortes de gaming_hours: tercil 1: observado=2.8; esperado=2.8; OK
cortes de gaming_hours: tercil 2: observado=5.46; esperado=5.46; OK


### 3.6 Redondeo
Después de crear `flag_ajustado`, se redondean a dos decimales las columnas float. `grades_original` se excluye para conservar la evidencia original sin redondear.

In [7]:
columnas_float = df.select_dtypes(include=['float64', 'float32']).columns.tolist()
columnas_float = [columna for columna in columnas_float if columna != 'grades_original']
for columna in columnas_float:
    df[columna] = df[columna].round(2)
excesos_decimales = int(sum(((df[columna] - df[columna].round(2)).abs() > 1e-9).sum() for columna in columnas_float))
comparar('valores float con más de dos decimales después del redondeo', excesos_decimales, 0)
registrar('3.6', 'Redondeo', ', '.join(columnas_float), 'Valores float con más de dos decimales', cantidad_ajustados, 'Redondear a dos decimales', 'Facilita lectura y mantiene grades_original sin alterar.')

valores float con más de dos decimales después del redondeo: observado=0; esperado=0; OK


### 3.7 Descarte de columnas
Antes de descartar se calcula la evidencia. `student_id` es un identificador; `reaction_time_ms` es redundante con horas de juego y no corresponde al objetivo; `gaming_genre` queda fuera de alcance; `addiction_score` tiene escala no documentada, valores negativos imposibles y posible redundancia. Se conservan explícitamente las variables académicas, demográficas, de uso y las notas.

In [8]:
student_id_secuencial = bool((df_raw['student_id'] == np.arange(1, n0 + 1)).all())
corr_reaction = df_raw['gaming_hours'].corr(df_raw['reaction_time_ms'])
corr_addiction = df_raw['gaming_hours'].corr(df_raw['addiction_score'])
negativos_addiction = int((df_raw['addiction_score'] < 0).sum())
print(f'student_id es fila + 1: {student_id_secuencial}')
print(f'Correlación gaming_hours/reaction_time_ms: {corr_reaction:.4f}')
print(f'Correlación gaming_hours/addiction_score: {corr_addiction:.4f}')
print(f'addiction_score negativos en el original: {negativos_addiction}')
comparar('correlación gaming_hours/reaction_time_ms', float(corr_reaction), -0.94, tolerancia=0.01)
comparar('correlación gaming_hours/addiction_score', float(corr_addiction), 0.91, tolerancia=0.01)
comparar('addiction_score negativos', negativos_addiction, 107)
registrar('3.7', 'Descarte de columnas', 'reaction_time_ms', 'Correlación con gaming_hours', 0, f'r={corr_reaction:.4f}', 'Redundante con gaming_hours y fuera del objetivo definido.')
registrar('3.7', 'Descarte de columnas', 'addiction_score', 'Valores negativos y correlación con gaming_hours', negativos_addiction, f'r={corr_addiction:.4f}; negativos={negativos_addiction}', 'Escala no documentada; los negativos son anomalías del original y no se corrigen porque la columna se descarta.')

COLUMNAS_DESCARTADAS = {
    'student_id': 'Identificador único sin información analítica; coincide con número de fila + 1.',
    'reaction_time_ms': 'Redundante con gaming_hours y no vinculada al objetivo del proyecto.',
    'gaming_genre': 'Fuera del alcance: el análisis se centra en horas de juego, no tipo de juego.',
    'addiction_score': 'Escala no documentada, valores negativos y casi redundante con gaming_hours.'
}
for columna, justificacion in COLUMNAS_DESCARTADAS.items():
    registrar('3.7', 'Descarte de columnas', columna, 'Columna fuera del dataset analítico final', n0, 'Descartar', justificacion)
df = df.drop(columns=list(COLUMNAS_DESCARTADAS))

COLUMNAS_CONSERVADAS = ['age', 'gender', 'gaming_hours', 'study_hours', 'sleep_hours', 'attendance', 'social_activity', 'device_usage', 'stress_level', 'grades']
columnas_finales = ['age', 'gender', 'gaming_hours', 'study_hours', 'sleep_hours', 'attendance', 'social_activity', 'device_usage', 'uso_no_gaming', 'stress_level', 'gaming_cat', 'grades', 'grades_original', 'flag_grades_gt100', 'flag_ajustado']
df = df[columnas_finales]
print(f'Columnas descartadas: {list(COLUMNAS_DESCARTADAS)}')
print(f'Columnas conservadas explícitamente: {COLUMNAS_CONSERVADAS}')

student_id es fila + 1: True
Correlación gaming_hours/reaction_time_ms: -0.9389
Correlación gaming_hours/addiction_score: 0.9086
addiction_score negativos en el original: 107
correlación gaming_hours/reaction_time_ms: observado=-0.9389431437392199; esperado=-0.94; OK
correlación gaming_hours/addiction_score: observado=0.9085989038090079; esperado=0.91; OK
addiction_score negativos: observado=107; esperado=107; OK
Columnas descartadas: ['student_id', 'reaction_time_ms', 'gaming_genre', 'addiction_score']
Columnas conservadas explícitamente: ['age', 'gender', 'gaming_hours', 'study_hours', 'sleep_hours', 'attendance', 'social_activity', 'device_usage', 'stress_level', 'grades']


### 3.8 Validaciones finales
Estas son las validaciones estructurales finales. A diferencia de las revisiones previas, aquí se usan `assert` porque el entregable debe cumplir exactamente el contrato definido.

In [9]:
columnas_finales = ['age', 'gender', 'gaming_hours', 'study_hours', 'sleep_hours', 'attendance', 'social_activity', 'device_usage', 'uso_no_gaming', 'stress_level', 'gaming_cat', 'grades', 'grades_original', 'flag_grades_gt100', 'flag_ajustado']
assert len(df) == n0
assert not df.isna().any().any()
assert df['grades'].between(0, 100).all()
assert (df['uso_no_gaming'] >= 0).all()
assert not df.duplicated().any()
assert df.columns.tolist() == columnas_finales
assert isinstance(df['stress_level'].dtype, pd.CategoricalDtype) and df['stress_level'].cat.ordered
assert isinstance(df['gaming_cat'].dtype, pd.CategoricalDtype) and df['gaming_cat'].cat.ordered
assert df['flag_grades_gt100'].dtype == bool
assert df['flag_ajustado'].dtype == bool
assert df_raw.shape == (8000, 14)
print('Validaciones finales: OK')

Validaciones finales: OK


### 3.9 Exportación y resumen
Se exportan el dataset limpio y la bitácora. Luego se vuelve a leer el CSV para comprobar que el entregable conserva la forma y las columnas esperadas.

In [10]:
bitacora = pd.DataFrame(registros_bitacora, columns=['n_paso', 'paso', 'columna', 'revision_o_problema', 'registros_afectados', 'decision', 'fundamento', 'filas_despues'])
df.to_csv(CLEAN_PATH, index=False, encoding='utf-8')
bitacora.to_csv(BITACORA_PATH, index=False, encoding='utf-8')

df_guardado = pd.read_csv(CLEAN_PATH)
assert df_guardado.shape == (n0, len(columnas_finales))
assert df_guardado.columns.tolist() == columnas_finales
print(f'Dataset guardado en: {CLEAN_PATH}')
print(f'Bitácora guardada en: {BITACORA_PATH}')
print(f'Filas: {n0} antes / {len(df)} después')
print(f'Columnas: {len(df_raw.columns)} antes / {len(df.columns)} después')
print(f'Columnas eliminadas: {list(COLUMNAS_DESCARTADAS)}')
print('Columnas creadas: uso_no_gaming, gaming_cat, grades_original, flag_grades_gt100, flag_ajustado')
print('Bitácora completa:')
display(bitacora)

Dataset guardado en: ..\data\processed\gaming_academic_clean.csv
Bitácora guardada en: ..\output\bitacora_limpieza.csv
Filas: 8000 antes / 8000 después
Columnas: 14 antes / 15 después
Columnas eliminadas: ['student_id', 'reaction_time_ms', 'gaming_genre', 'addiction_score']
Columnas creadas: uso_no_gaming, gaming_cat, grades_original, flag_grades_gt100, flag_ajustado
Bitácora completa:


,n_paso,paso,columna,revision_o_problema,registros_afectados,decision,fundamento,filas_despues
0,3.1,Revisiones previas,student_id,Valores faltantes,0,No imputar,Se conservan las filas y se documentan los fal...,8000
1,3.1,Revisiones previas,age,Valores faltantes,0,No imputar,Se conservan las filas y se documentan los fal...,8000
2,3.1,Revisiones previas,gender,Valores faltantes,0,No imputar,Se conservan las filas y se documentan los fal...,8000
3,3.1,Revisiones previas,gaming_hours,Valores faltantes,0,No imputar,Se conservan las filas y se documentan los fal...,8000
4,3.1,Revisiones previas,study_hours,Valores faltantes,0,No imputar,Se conservan las filas y se documentan los fal...,8000
5,3.1,Revisiones previas,sleep_hours,Valores faltantes,0,No imputar,Se conservan las filas y se documentan los fal...,8000
6,3.1,Revisiones previas,attendance,Valores faltantes,0,No imputar,Se conservan las filas y se documentan los fal...,8000
7,3.1,Revisiones previas,gaming_genre,Valores faltantes,0,No imputar,Se conservan las filas y se documentan los fal...,8000
8,3.1,Revisiones previas,social_activity,Valores faltantes,0,No imputar,Se conservan las filas y se documentan los fal...,8000
9,3.1,Revisiones previas,device_usage,Valores faltantes,0,No imputar,Se conservan las filas y se documentan los fal...,8000


#### Recarga posterior del dataset limpio
Las secciones 4 y 5 trabajarán directamente con el DataFrame `df` limpio que queda en memoria, sin recargarlo desde el CSV. El CSV se exporta como entregable y para reproducibilidad.

Si se necesita recargarlo en otra sesión, se deben restaurar las categorías ordenadas así:

```python
df = pd.read_csv('../data/processed/gaming_academic_clean.csv')
df['stress_level'] = pd.Categorical(df['stress_level'], categories=['Low', 'Medium', 'High'], ordered=True)
df['gaming_cat'] = pd.Categorical(df['gaming_cat'], categories=['Bajo', 'Medio', 'Alto'], ordered=True)
```

## Sección 4. Análisis univariado y bivariado
PENDIENTE.

## Sección 5. Análisis descriptivo y exploratorio
PENDIENTE.

## Sección 6. Conclusión de la Fase 1
PENDIENTE.